# CodeQA: frozen CodeBERT + pretrained BART decoder (VPS)

Eksperimen lanjutan, bukan baseline CodeQA dari paper. CodeBERT dibekukan; bridge dan decoder BART dilatih. Decoder sudah pretrained, tetapi pasangan CodeBERT–BART belum pernah dilatih bersama.

Preset `vps`: 56.085 train, 1.000 dev tetap, maksimal 20 epoch, patience 5; checkpoint terbaik dipilih dari dev BLEU. Test 7.000 hanya dibuka setelah training selesai. Source 400, target 32, effective batch 64, microbatch 2, FP16 CUDA. Dev/test menggunakan greedy, berbeda dengan beam 10 baseline CodeBERT. Skor bukan perbandingan arsitektur yang sepenuhnya terkontrol.

Jalankan dari folder paket VPS. Untuk training yang bisa ditinggal, gunakan runner terminal di tmux; notebook ini memakai runner yang sama. Tidak perlu mengulang baseline sebelumnya atau menyalin dataset baru.


In [ ]:
from pathlib import Path
import json
from codebert_bart_training import Settings, run

assert Path("data/python/train/train.code").is_file(), "Buka notebook dari folder paket VPS"
PROFILE = "vps"  # ganti ke smoke untuk 8 train / 4 dev / 4 test / 1 epoch
RUN_TAG = "v1"
settings = Settings(profile=PROFILE, run_tag=RUN_TAG, micro_batch=2, max_session_seconds=0)
settings


## Training atau resume

Perintah/konfigurasi/tag yang sama melanjutkan `last.pt` beserta optimizer, scheduler, scaler dan RNG. Checkpoint ditulis setiap sekitar lima menit dan akhir epoch. Jika interupsi, pekerjaan setelah checkpoint terakhir diulang. Jangan jalankan notebook dan runner terminal untuk tag yang sama secara bersamaan.

Mengulang sel pada run yang sudah selesai akan mengulang evaluasi test, bukan training. Untuk membaca hasil selesai, langsung buka `results.json` atau jalankan sel pembacaan hasil di bawah.


In [ ]:
OUT = run(settings)
print("Run directory:", OUT)


In [ ]:
# Sel ini bisa dijalankan tanpa menjalankan sel training.
candidates = list(Path("codebert_bart_results").glob(f"{PROFILE}_{RUN_TAG}_*"))
assert len(candidates) == 1, f"Run belum tersedia atau ambigu: {candidates}"
OUT = candidates[0]
path = OUT / "results.json"
print(path.read_text() if path.exists() else (OUT / "status.json").read_text())
